# Script di addestramento per modelli di detection

In [ ]:
import os
from ultralytics import YOLO
import torch
import gc

# Ottimizzazioni generali
gc.collect()
torch.cuda.empty_cache()
torch.backends.cudnn.benchmark = True 
torch.backends.cudnn.enabled = True  

# Modificare i percorsi secondo necessità
DATASET_DIR = os.path.abspath("./working/data_sampled") 
YAML_PATH = os.path.join(DATASET_DIR, "data.yaml")

# Modello base scaricato
BASE_MODEL = "./detection models/others/yolo11m.pt"  

PROJECT_DIR = "./runs/detect"
RUN_NAME = "./detection models/ft_1920_yolo11m"

print(f"Caricamento modello base: {BASE_MODEL}")
model = YOLO(BASE_MODEL)

try:
    model.train(
        data=YAML_PATH,
        epochs=50,          
        imgsz=1920,         
        batch=1,            
        freeze=0,           
        patience=10,
        optimizer='auto',   
        project=PROJECT_DIR,
        name=RUN_NAME,
        device=0,
        amp=True,
        close_mosaic=10,    
        workers=2,
        plots=True
    )
except torch.cuda.OutOfMemoryError:
    print("\nERRORE VRAM")

print(f"Training Completato!")
    

# Script di addestramento per modelli di re-identificazione

In [ ]:
import os
import copy
import gc
import numpy as np
from collections import defaultdict
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Sampler
from torchvision import datasets, transforms

# File proveniente dalla libreria torchreid
from osnet_ain import osnet_ain_x1_0

# Modificare i percorsi secondo necessità
DATA_DIR = "./reid_dataset"
IMG_SIZE = (256, 128)

# Checkpoint 
CHECKPOINT_PATH = "./reid_models/others/osnet_ain_x1_0_msmt17.pth" 
SAVE_PATH = "./reid_models/osnet_ain_x1_0_msmt17.pth"

P = 16          # Identità per batch
K = 4           # Immagini per identità
BATCH_SIZE = P * K

EPOCHS = 50     
LR = 1e-4       
NUM_WORKERS = 4 

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class RandomIdentitySampler(Sampler):
    """
    Campiona P identità e K immagini per ciascuna.
    """
    def __init__(self, data_source, batch_size, num_instances):
        self.data_source = data_source
        self.batch_size = batch_size
        self.num_instances = num_instances
        self.index_dic = defaultdict(list)
        
        # Raggruppamento indici per identità 
        for index, (_, pid) in enumerate(data_source.imgs):
            self.index_dic[pid].append(index)
            
        self.pids = list(self.index_dic.keys())
        self.num_identities = len(self.pids)

    def __iter__(self):
        batch_idxs_dict = defaultdict(list)

        for pid in self.pids:
            idxs = self.index_dic[pid]
            if len(idxs) < self.num_instances:
                idxs = np.random.choice(idxs, size=self.num_instances, replace=True)
            
            np.random.shuffle(idxs)
            batch_idxs_dict[pid] = idxs

        avai_pids = copy.deepcopy(self.pids)
        final_idxs = []

        # Costruzione dei batch 
        while len(avai_pids) >= self.batch_size // self.num_instances:
            selected_pids = np.random.choice(avai_pids, self.batch_size // self.num_instances, replace=False)
            
            for pid in selected_pids:
                # Estrazione primi K indici
                batch_idxs = batch_idxs_dict[pid][:self.num_instances]
                final_idxs.extend(batch_idxs)
                
                # Rimozione indici usati
                batch_idxs_dict[pid] = batch_idxs_dict[pid][self.num_instances:]
                
                if len(batch_idxs_dict[pid]) < self.num_instances:
                    avai_pids.remove(pid)

        return iter(final_idxs)

    def __len__(self):
        return self.num_identities * self.num_instances

class BatchHardTripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)

    def forward(self, inputs, targets):
        n = inputs.size(0)
        
        # Calcolo distanze euclidee a coppie 
        dist = torch.pow(inputs, 2).sum(dim=1, keepdim=True).expand(n, n)
        dist = dist + dist.t()
        dist.addmm_(inputs, inputs.t(), beta=1, alpha=-2)
        dist = dist.clamp(min=1e-12).sqrt()

        # Maschera per positivi 
        mask = targets.expand(n, n).eq(targets.expand(n, n).t())
        
        # Hardest Positive: max distanza tra i positivi
        dist_ap, _ = torch.max(dist * mask.float(), dim=1)
        
        # Hardest Negative: min distanza tra i negativi
        # Si aggiunge 1e6 ai positivi per escluderli dal minimo
        dist_an, _ = torch.min(dist + (mask.float() * 1e6), dim=1)

        y = torch.ones_like(dist_an)
        return self.ranking_loss(dist_an, dist_ap, y)

def extract_features(model, loader):
    model.eval()
    feats, pids = [], []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            f = model(imgs)
            f = F.normalize(f, p=2, dim=1) 
            
            feats.append(f.cpu())
            pids.append(labels.cpu())
    
    return torch.cat(feats), torch.cat(pids)

def evaluate(model, val_loader):
    print("Estrazione features validazione...")
    features, labels = extract_features(model, val_loader)
    
    # STRATEGIA QUERY / GALLERY 
    # Si prende 1 immagine per persona come Query, il resto Gallery
    query_mask = np.zeros(len(labels), dtype=bool)
    unique_labels = torch.unique(labels)
    
    for label in unique_labels:
        idxs = (labels == label).nonzero(as_tuple=True)[0]
        query_mask[idxs[0]] = True 
        
    query_feats = features[query_mask]
    query_lbls = labels[query_mask]
    gallery_feats = features[~query_mask]
    gallery_lbls = labels[~query_mask]
    
    print(f"Eval Split: {len(query_feats)} Query vs {len(gallery_feats)} Gallery")
    
    m, n = query_feats.shape[0], gallery_feats.shape[0]
    
    q_sq = torch.pow(query_feats, 2).sum(dim=1, keepdim=True).expand(m, n)
    g_sq = torch.pow(gallery_feats, 2).sum(dim=1, keepdim=True).expand(n, m).t()
    
    distmat = q_sq + g_sq
    distmat.addmm_(query_feats, gallery_feats.t(), beta=1, alpha=-2)
    distmat = distmat.cpu().numpy()
    
    # Calcolo Metriche
    cmc = np.zeros(len(gallery_lbls))
    ap = 0.0
    
    gallery_lbls_np = gallery_lbls.numpy()
    query_lbls_np = query_lbls.numpy()
    
    for i in range(m):
        indices = np.argsort(distmat[i])
        matches = (gallery_lbls_np[indices] == query_lbls_np[i])
        
        if not np.any(matches): continue
            
        # CMC
        cmc_i = matches.cumsum()
        cmc_i[cmc_i > 1] = 1
        cmc += cmc_i
        
        # AP
        num_rel = matches.sum()
        tmp_cmc = matches.cumsum()
        tmp_cmc = [x / (j + 1.) for j, x in enumerate(tmp_cmc)]
        tmp_cmc = np.asarray(tmp_cmc) * matches
        ap += tmp_cmc.sum() / num_rel

    cmc /= m
    mAP = ap / m
    
    return {
        "mAP": mAP * 100,
        "Rank-1": cmc[0] * 100,
        "Rank-5": cmc[4] * 100
    }

def main():
    torch.backends.cudnn.benchmark = True
    
    train_tf = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.Pad(10),
        transforms.RandomCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        transforms.RandomErasing(p=0.5)
    ])

    val_tf = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    train_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), train_tf)
    val_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), val_tf)

    sampler = RandomIdentitySampler(train_ds, BATCH_SIZE, K)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, sampler=sampler,
        num_workers=NUM_WORKERS, pin_memory=True
    )
    
    val_loader = DataLoader(
        val_ds, batch_size=64, shuffle=False, num_workers=NUM_WORKERS
    )

    print(f"Dataset caricato: {len(train_ds)} training imgs.")
    
    num_classes = len(train_ds.classes)
    model = osnet_ain_x1_0(num_classes=num_classes, loss="triplet", pretrained=False)

    if os.path.exists(CHECKPOINT_PATH):
        print(f"Caricamento pesi da: {CHECKPOINT_PATH}")
        ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
        
        state_dict = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt
        
        missing, _ = model.load_state_dict(state_dict, strict=False)
        print(f"Pesi caricati. Missing keys: {len(missing)}")
    else:
        print(f"File {CHECKPOINT_PATH} non trovato! Inizio da zero.")

    model.to(DEVICE)

    ce_loss = nn.CrossEntropyLoss()
    tri_loss = BatchHardTripletLoss()
    
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[15, 30], gamma=0.1)

    best_r1 = 0.0
    
    for epoch in range(EPOCHS):
        model.train()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        
        loss_avg = 0.0
        
        for i, (imgs, labels) in enumerate(pbar):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            
            scores, feats = model(imgs)
            
            loss = ce_loss(scores, labels) + tri_loss(feats, labels)
            
            loss.backward()
            optimizer.step()
            
            loss_avg += loss.item()
            if i % 10 == 0:
                pbar.set_postfix(loss=f"{loss.item():.3f}", lr=f"{optimizer.param_groups[0]['lr']:.1e}")

        scheduler.step()
        
        torch.cuda.empty_cache()
        gc.collect()
        
        metrics = evaluate(model, val_loader)
        print(f"Result Ep {epoch+1}: mAP {metrics['mAP']:.1f}% | R1 {metrics['Rank-1']:.1f}% | R5 {metrics['Rank-5']:.1f}%")
        
        if metrics['Rank-1'] > best_r1:
            best_r1 = metrics['Rank-1']
            torch.save(model.state_dict(), SAVE_PATH)
            print(f"Modello salvato in {SAVE_PATH}")

    print("Fine Training.")

if __name__ == "__main__":
    gc.collect()
    torch.cuda.empty_cache()
    main()

# Convertitore modelli .pth in modelli .pt

In [ ]:
import torch

# Modificare i percorsi secondo necessità
# Definizione dei percorsi
input_path = './converter/osnet_ain_x1_0.pth'
output_path = './converter/osnet_ain_x1_0.pt'

# Caricamento del file originale 
checkpoint = torch.load(input_path, map_location='cpu', weights_only=True)

# Estrazione dei pesi 
if 'state_dict' in checkpoint:
    weights = checkpoint['state_dict']
elif 'model' in checkpoint:
    weights = checkpoint['model']
else:
    weights = checkpoint

# Pulizia
new_weights = {}
for k, v in weights.items():
    name = k.replace("module.", "") 
    new_weights[name] = v

torch.save(new_weights, output_path)

print(f"Convertito con successo: {output_path}")

Convertito con successo: ./reid_models/others/osnet_ain_x1_0_imagenet.pt
